In [3]:
import pandas as pd

df = pd.read_csv("../data/cleaned_emails.csv")

In [5]:
df.shape

(3978, 8)

In [6]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (3182,)
Testing: (796,)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (3182, 20000)
Test TF-IDF shape: (796, 20000)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [9]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "spam").astype(int),
        y_prob
    )
)

Accuracy: 0.9660804020100503

Confusion Matrix:
[[523   5]
 [ 22 246]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      0.99      0.97       528
        spam       0.98      0.92      0.95       268

    accuracy                           0.97       796
   macro avg       0.97      0.95      0.96       796
weighted avg       0.97      0.97      0.97       796

ROC-AUC: 0.9959153098145636


In [11]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

coefficients = model.coef_[0]

top_spam = np.argsort(coefficients)[-20:][::-1]
top_ham = np.argsort(coefficients)[:20]

print("Top SPAM indicators:\n")

for i in top_spam:
    print(
        feature_names[i],
        round(coefficients[i], 4)
    )

print("\nTop HAM indicators:\n")

for i in top_ham:
    print(
        feature_names[i],
        round(coefficients[i], 4)
    )

Top SPAM indicators:

your 4.5593
you 4.4086
font 3.9227
br 2.809
our 2.7843
sightings 2.5929
free 2.5788
we 2.2935
remove 2.2248
please 2.1867
will 1.762
center 1.7554
credit 1.676
align 1.6755
money 1.6346
this 1.6077
below 1.5475
receive 1.5281
000 1.519
email 1.4968

Top HAM indicators:

the -3.3281
cnet -3.0612
lockergnome -2.832
re -2.7095
2002 -2.6893
url -2.4809
date -2.4022
that -2.2346
rpm -2.0737
it -2.048
newsisfree -2.0256
clickthru -1.9615
but -1.9137
wrote -1.7064
razor -1.6446
09 -1.6071
zdnet -1.5153
perl -1.4145
supplied -1.4088
01 -1.3859


# Observations

The model has an ROC-AUC of 99.59% with a with an F1 score of 95% for spam, this is a peak performance by the model <br>
but the email contains many html-related terms and potentially ambiguous terms like the, you, your, so the model may be <br>
learning dataset-specific artifacts rather than general spam behavior.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_ngram = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000
)

X_train_tfidf = vectorizer_ngram.fit_transform(X_train)
X_test_tfidf = vectorizer_ngram.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (3182, 30000)
Test shape: (796, 30000)


In [13]:
model_ngram = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_ngram.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [14]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_pred = model_ngram.predict(X_test_tfidf)
y_prob = model_ngram.predict_proba(X_test_tfidf)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "spam").astype(int),
        y_prob
    )
)

Accuracy: 0.9736180904522613

Confusion Matrix:
[[525   3]
 [ 18 250]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       528
        spam       0.99      0.93      0.96       268

    accuracy                           0.97       796
   macro avg       0.98      0.96      0.97       796
weighted avg       0.97      0.97      0.97       796

ROC-AUC: 0.996445330167345


Model improved after using ngram, the model is making fewer mistakes in both directions.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

print("Train shape:", X_train_char.shape)
print("Test shape:", X_test_char.shape)

Train shape: (3182, 30000)
Test shape: (796, 30000)


In [19]:
char_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

char_model.fit(X_train_char, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [20]:
y_pred_char = char_model.predict(X_test_char)
y_prob_char = char_model.predict_proba(X_test_char)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_char))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_char))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_char))

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "spam").astype(int),
        y_prob_char
    )
)

Accuracy: 0.9623115577889447

Confusion Matrix:
[[518  10]
 [ 20 248]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      0.98      0.97       528
        spam       0.96      0.93      0.94       268

    accuracy                           0.96       796
   macro avg       0.96      0.95      0.96       796
weighted avg       0.96      0.96      0.96       796

ROC-AUC: 0.9942828471279963


The character-level TF-IDF performed worse on this dataset compared to word-level TF-IDF

In [21]:
from scipy.sparse import hstack

X_train_hybrid = hstack([
    X_train_tfidf,
    X_train_char
])

X_test_hybrid = hstack([
    X_test_tfidf,
    X_test_char
])

print("Hybrid train shape:", X_train_hybrid.shape)
print("Hybrid test shape:", X_test_hybrid.shape)

Hybrid train shape: (3182, 60000)
Hybrid test shape: (796, 60000)


In [22]:
hybrid_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

hybrid_model.fit(X_train_hybrid, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [23]:
y_pred_hybrid = hybrid_model.predict(X_test_hybrid)
y_prob_hybrid = hybrid_model.predict_proba(X_test_hybrid)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_hybrid))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hybrid))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hybrid))

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "spam").astype(int),
        y_prob_hybrid
    )
)

Accuracy: 0.9723618090452262

Confusion Matrix:
[[520   8]
 [ 14 254]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      0.98      0.98       528
        spam       0.97      0.95      0.96       268

    accuracy                           0.97       796
   macro avg       0.97      0.97      0.97       796
weighted avg       0.97      0.97      0.97       796

ROC-AUC: 0.9972226933514248


# Comparison

| Model | Accuracy | Spam Precision | Spam Recall | Spam F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Word TF-IDF (1,1) | 96.61% | 98% | 92% | 95% | 99.59% |
| Word TF-IDF (1,2) | 97.36% | 99% | 93% | 96% | 99.64% |
| Character TF-IDF (3,5) | 96.23% | 96% | 93% | 94% | 99.43% |
| Word + Character | 97.24% | 97% | 95% | 96% | 99.72% |

The hybrid model ( Word + Character )  has the best Overall Performance.

In [24]:
import re
import pandas as pd

def extract_email_features(df):
    features = pd.DataFrame(index=df.index)

    subject = df["subject"].fillna("")
    body = df["body"].fillna("")
    text = df["text"].fillna("")

    features["subject_length"] = subject.str.len()
    features["body_length"] = body.str.len()
    features["text_length"] = text.str.len()

    features["word_count"] = text.str.split().str.len()

    features["url_count"] = text.str.count(
        r"https?://|www\."
    )

    features["html_count"] = text.str.count(
        r"<[^>]+>"
    )

    features["exclamation_count"] = text.str.count("!")
    features["question_count"] = text.str.count(r"\?")
    features["digit_count"] = text.str.count(r"\d")

    features["uppercase_count"] = text.apply(
        lambda x: sum(c.isupper() for c in x)
    )

    return features

In [25]:
email_features = extract_email_features(df)

print(email_features.head())
print(email_features.shape)

   subject_length  body_length  text_length  word_count  url_count  \
0              24         1604         1627         229          1   
1              25          899          920         114          2   
2              25         1751         1772         255          2   
3              37         1127         1163         171          3   
4              20         1297         1316         210          1   

   html_count  exclamation_count  question_count  digit_count  uppercase_count  
0           2                  0               0          116               49  
1           0                  2               1           12               46  
2           0                  2               0           13               97  
3           0                  0               0           27               51  
4           0                  1               2           14               26  
(3978, 10)


In [27]:
email_train = email_features.loc[X_train.index]
email_test = email_features.loc[X_test.index]

print("Email train:", email_train.shape)
print("Email test:", email_test.shape)

Email train: (3182, 10)
Email test: (796, 10)


In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

email_train_scaled = scaler.fit_transform(email_train)
email_test_scaled = scaler.transform(email_test)

In [29]:
from scipy.sparse import csr_matrix, hstack

email_train_sparse = csr_matrix(email_train_scaled)
email_test_sparse = csr_matrix(email_test_scaled)

In [30]:
X_train_email = hstack([
    X_train_tfidf,
    email_train_sparse
])

X_test_email = hstack([
    X_test_tfidf,
    email_test_sparse
])

print("Train shape:", X_train_email.shape)
print("Test shape:", X_test_email.shape)

Train shape: (3182, 30010)
Test shape: (796, 30010)


In [31]:
email_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

email_model.fit(X_train_email, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [32]:
y_pred_email = email_model.predict(X_test_email)
y_prob_email = email_model.predict_proba(X_test_email)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_email))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_email))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_email))

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "spam").astype(int),
        y_prob_email
    )
)

Accuracy: 0.9610552763819096

Confusion Matrix:
[[520   8]
 [ 23 245]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.96      0.98      0.97       528
        spam       0.97      0.91      0.94       268

    accuracy                           0.96       796
   macro avg       0.96      0.95      0.96       796
weighted avg       0.96      0.96      0.96       796

ROC-AUC: 0.9896893374038896


In [33]:
email_feature_names = email_features.columns

email_coefficients = email_model.coef_[0][-len(email_feature_names):]

email_importance = pd.DataFrame({
    "Feature": email_feature_names,
    "Coefficient": email_coefficients
})

print(
    email_importance.sort_values(
        "Coefficient",
        ascending=False
    ).to_string(index=False)
)

          Feature  Coefficient
exclamation_count     1.046944
  uppercase_count     0.885652
      digit_count     0.097769
   subject_length     0.062109
       word_count     0.014556
      body_length    -0.195966
      text_length    -0.214902
       html_count    -0.365264
   question_count    -0.956699
        url_count    -1.813594


In [42]:
y_train_binary = (y_train == "spam").astype(int)

In [38]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

word_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

word_cv = cross_validate(
    word_model,
    X_train_tfidf,
    y_train_binary,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    n_jobs=-1
)

print("Word TF-IDF CV")
print("Accuracy:", word_cv["test_accuracy"])
print("Mean Accuracy:", word_cv["test_accuracy"].mean())

print("Precision:", word_cv["test_precision"])
print("Mean Precision:", word_cv["test_precision"].mean())

print("Recall:", word_cv["test_recall"])
print("Mean Recall:", word_cv["test_recall"].mean())

print("F1:", word_cv["test_f1"])
print("Mean F1:", word_cv["test_f1"].mean())

print("ROC-AUC:", word_cv["test_roc_auc"])
print("Mean ROC-AUC:", word_cv["test_roc_auc"].mean())

Word TF-IDF CV
Accuracy: [0.96703297 0.95918367 0.97955975 0.96226415 0.96540881]
Mean Accuracy: 0.9666898689809742
Precision: [0.99489796 0.97969543 0.9902439  0.97979798 0.98484848]
Mean Precision: 0.9858967515482486
Recall: [0.90697674 0.89767442 0.94859813 0.90654206 0.91121495]
Mean Recall: 0.9142012605955226
F1: [0.94890511 0.9368932  0.96897375 0.94174757 0.94660194]
Mean F1: 0.948624314990472
ROC-AUC: [0.99888681 0.99609831 0.99816185 0.99635691 0.99269168]
Mean ROC-AUC: 0.9964391112173224


In [39]:
hybrid_cv = cross_validate(
    hybrid_model,
    X_train_hybrid,
    y_train_binary,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    n_jobs=-1
)

print("\nWord + Character TF-IDF CV")
print("Accuracy:", hybrid_cv["test_accuracy"])
print("Mean Accuracy:", hybrid_cv["test_accuracy"].mean())

print("Precision:", hybrid_cv["test_precision"])
print("Mean Precision:", hybrid_cv["test_precision"].mean())

print("Recall:", hybrid_cv["test_recall"])
print("Mean Recall:", hybrid_cv["test_recall"].mean())

print("F1:", hybrid_cv["test_f1"])
print("Mean F1:", hybrid_cv["test_f1"].mean())

print("ROC-AUC:", hybrid_cv["test_roc_auc"])
print("Mean ROC-AUC:", hybrid_cv["test_roc_auc"].mean())


Word + Character TF-IDF CV
Accuracy: [0.9811617  0.97802198 0.98584906 0.97169811 0.97955975]
Mean Accuracy: 0.9792581183416763
Precision: [0.99512195 0.97183099 0.99043062 0.97572816 0.98550725]
Mean Precision: 0.9837237921722384
Recall: [0.94883721 0.9627907  0.96728972 0.93925234 0.95327103]
Mean Recall: 0.9542881982177788
F1: [0.97142857 0.96728972 0.9787234  0.95714286 0.96912114]
Mean F1: 0.9687411385190867
ROC-AUC: [0.99909622 0.99677064 0.99809541 0.99588076 0.99572574]
Mean ROC-AUC: 0.997113753346429


# Cross Validation Comparison

| Metric | Word TF-IDF | Word + Character TF-IDF |
|---|---:|---:|
| Accuracy | 96.67% | **97.93%** |
| Precision | **98.59%** | 98.37% |
| Spam Recall | 91.42% | **95.43%** |
| Spam F1 | 94.86% | **96.87%** |
| ROC-AUC | 99.64% | **99.71%** |

# Final Model

In [48]:
y_train_binary = (y_train == "spam").astype(int)
y_test_binary = (y_test == "spam").astype(int)

print("y_train_binary:", y_train_binary.shape)
print("y_test_binary:", y_test_binary.shape)

y_train_binary: (3182,)
y_test_binary: (796,)


In [49]:
final_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

final_model.fit(
    X_train_hybrid,
    y_train_binary
)

y_pred_final = final_model.predict(X_test_hybrid)
y_prob_final = final_model.predict_proba(X_test_hybrid)[:, 1]

In [50]:
print("Final Accuracy:",
      accuracy_score(y_test_binary, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_binary, y_pred_final))

print("\nClassification Report:")
print(classification_report(
    y_test_binary,
    y_pred_final,
    target_names=["ham", "spam"]
))

print("Final ROC-AUC:",
      roc_auc_score(y_test_binary, y_prob_final))

Final Accuracy: 0.9723618090452262

Confusion Matrix:
[[520   8]
 [ 14 254]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      0.98      0.98       528
        spam       0.97      0.95      0.96       268

    accuracy                           0.97       796
   macro avg       0.97      0.97      0.97       796
weighted avg       0.97      0.97      0.97       796

Final ROC-AUC: 0.9972226933514248
